# CTR Opportunity Scoring for Search Content Review

## Abstract

This project asks whether current search visibility signals can prioritize pages that may under-capture clicks. I used the FlyRank internship warehouse's March 2026 daily content-performance partition, aggregated available Search Console observations to client-content-month rows, and compared a transparent CTR-gap baseline with Logistic Regression. On a client-grouped April outcome split, the baseline measured Precision@10 of 0.40 and Logistic Regression measured 0.30. The result is directional decision-support evidence: a human should inspect the highest-scoring pages, and no metadata change should be automated from the score alone.

## 1) Question

Can a small, leakage-aware feature set prioritize visible pages whose CTR may improve with metadata or content review? The action is a ranked review queue, not a causal intervention claim.

## 2) Data

- Release: FlyRank Internship Warehouse, pseudonymized and gated.
- Table: `fact_content_daily_performance`; `dim_content` is available for future content metadata joins.
- Feature window: March 2026. Outcome window: April 2026.
- Raw grain: one client-content-report-date row. Decision grain: one client-content-month after aggregation.
- Excluded: future outcomes from features, client/content identity interpretation, and any public client-identifying output.
- Data credit: Built on the FlyRank ML Internship dataset, with source credit to [FlyRank](https://flyrank.ai).

## 3) Methodology

The baseline score is positive CTR gap versus a position benchmark multiplied by `log1p(impressions)`. The model is Logistic Regression over impressions, CTR, average position, organic sessions, and CTR gap. The future label is April CTR improvement. Validation groups by pseudonymized client so a client does not appear in both train and test.

The Week 4 baseline and Week 5 model use the same held-out clients and Precision@10. The Week 6 audit checks random versus grouped validation and removes future-window and label-derived fields from features.

In [1]:
from pathlib import Path
import json
import pandas as pd

workspace_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'work' / 'outputs').exists()), Path.cwd())
results = pd.DataFrame({
    'method': ['Week 4 baseline', 'Logistic Regression'],
    'precision_at_10': [0.40, 0.30],
})
playbook_receipt = json.loads((workspace_root / 'work' / 'outputs' / 'action_playbook_metrics.json').read_text())
print(results.to_string(index=False))
print(f"Playbook rows: {playbook_receipt['rows']:,}")
results

             method  precision_at_10
    Week 4 baseline              0.4
Logistic Regression              0.3
Playbook rows: 176,738


,method,precision_at_10
0,Week 4 baseline,0.4
1,Logistic Regression,0.3


## 4) Results and interpretation

On the same grouped test design, the transparent baseline outperformed the Logistic Regression model at the top-ten queue metric. The model's measured result does not justify complexity, so the baseline remains the preferred action-scoring reference. Errors include false positives caused by SERP context, position, seasonality, intent mismatch, and incomplete engagement coverage.

This is an observed comparison on one March-to-April slice, not evidence that the baseline causes future improvement.

## 5) Limitations

CTR gaps are observational and may reflect SERP features or search intent rather than weak metadata. The data is an unbalanced panel, GA4 coverage is incomplete, and one future month is not enough to establish stability. Hash keys are pseudonymous and are not interpreted as identities. Human review remains mandatory.

## 6) Ranked recommendations

1. Review the highest baseline-scoring pages for title, description, SERP features, and intent alignment.
2. Require a reviewer to reject rows where position, seasonality, or page intent explains the CTR gap.
3. Re-measure future Precision@10 across additional months before considering model replacement.
4. Do not automate publishing, deletion, redirects, or claims of causal lift.

## 7) Reproducibility and credit

- Week 3 contract: `work/notebooks/w03_data_contract.ipynb`
- Week 4 baseline: `work/notebooks/w04_baseline_score.ipynb`
- Week 5 model: `work/notebooks/w05_model.ipynb`
- Week 6 audit: `work/notebooks/w06_validation_audit.ipynb`
- Week 7 playbook: `work/notebooks/w07_action_playbook.ipynb`
- Generated JSON receipts live under `work/outputs/`; raw data is excluded from Git.

Built on the FlyRank ML Internship dataset. Data source: [flyrank.ai](https://flyrank.ai).

## 8) Five-minute demo outline

1. Show the question: which visible pages under-capture clicks?
2. Show the March-to-April time boundary and client-grouped split.
3. Show one signal bucket table and the Week 4 baseline queue.
4. Show the honest result: baseline Precision@10 0.40 versus model 0.30.
5. End with one recommendation: human-review the highest CTR-gap pages.

### Shareable methodology cut

I built a leakage-aware CTR opportunity scorer on pseudonymized FlyRank search-performance data, compared a transparent baseline with Logistic Regression, and validated the comparison on held-out clients.

### Employer-facing summary

Built a real-data search opportunity pipeline using FlyRank's pseudonymized warehouse, from page-month feature engineering through baseline scoring, grouped model validation, and a human-reviewed action playbook. The measured baseline outperformed the first Logistic Regression model on held-out-client Precision@10, so I kept the simpler rule and documented the limits instead of rewarding complexity.